In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
store_cleaned = pd.read_csv(r'E:\data-science-projects\Rossmann Store Sales\data\processed\store_cleaned.csv')
train_cleaned = pd.read_csv(r'E:\data-science-projects\Rossmann Store Sales\data\processed\train_cleaned.csv')
test_cleaned = pd.read_csv(r'E:\data-science-projects\Rossmann Store Sales\data\processed\test_cleaned.csv')

C:\Users\HP\AppData\Local\Temp\ipykernel_9140\434394389.py:2: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_cleaned = pd.read_csv(r'E:\data-science-projects\Rossmann Store Sales\data\processed\train_cleaned.csv')


In [4]:
train_df = train_cleaned.merge(store_cleaned, on='Store', how='left')

In [5]:
train_df['Date'] = pd.to_datetime(train_df['Date'])

train_df['Year'] = train_df['Date'].dt.year
train_df['Month'] = train_df['Date'].dt.month
train_df['Day'] = train_df['Date'].dt.day
train_df['DayOfWeek'] = train_df['Date'].dt.dayofweek
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week

In [6]:
train_df['CompetitionOpenSince'] = pd.to_datetime(
    train_df['CompetitionOpenSinceYear'].astype(str) + '-' +
    train_df['CompetitionOpenSinceMonth'].astype(str) + '-1',
    errors='coerce'
)

train_df['CompetitionDuration'] = (
    (train_df['Date'] - train_df['CompetitionOpenSince'])
    .dt.days
)

train_df['CompetitionDuration'] = train_df['CompetitionDuration'].fillna(0)

C:\Users\HP\AppData\Local\Temp\ipykernel_9140\643891701.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_df['CompetitionOpenSince'] = pd.to_datetime(


In [7]:
train_df['Promo2Since'] = pd.to_datetime(
    train_df['Promo2SinceYear'].astype(str) + '-' +
    train_df['Promo2SinceWeek'].astype(str) + '-1',
    format='%Y-%W-%w',
    errors='coerce'
)

train_df['PromoDuration'] = (
    (train_df['Date'] - train_df['Promo2Since'])
    .dt.days
)

train_df['PromoDuration'] = train_df['PromoDuration'].fillna(0)

In [8]:
month_map = {
    1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
    7:'Jul',8:'Aug',9:'Sept',10:'Oct',11:'Nov',12:'Dec'
}

train_df['MonthStr'] = train_df['Month'].map(month_map)

train_df['IsPromoMonth'] = train_df.apply(
    lambda x: 1 if x['MonthStr'] in str(x['PromoInterval']) else 0,
    axis=1
)

In [9]:
train_df = pd.get_dummies(train_df, columns=['StoreType','Assortment'], drop_first=True)

In [10]:
train_df['Sales_log'] = np.log1p(train_df['Sales'])

In [11]:
train_df.drop([
    'Date',
    'CompetitionOpenSince',
    'Promo2Since',
    'PromoInterval',
    'MonthStr'
], axis=1, inplace=True)

In [12]:
train_df.head()
train_df.isnull().sum()

Store                             0
DayOfWeek                         0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
Year                              0
Month                             0
Day                               0
WeekOfYear                        0
CompetitionDuration               0
PromoDuration                     0
IsPromoMonth                      0
StoreType_b                       0
StoreType_c                       0
StoreType_d                       0
Assortment_b                      0
Assortment_c                      0
Sales_log                         0
dtype: int64

In [13]:
train_df.to_csv(r'E:\data-science-projects\Rossmann Store Sales\data\processed\train_df.csv', index=False)